In [113]:
from neo4j import GraphDatabase
import networkx as nx
import pickle
import pandas as pd
import re
from tqdm import tqdm

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
print(f'base_dir is {base_dir}')
sys.path.append(os.path.dirname(base_dir))
from util.utils import extract_hgnc_name, load_graph

base_dir is /Users/yuxiaoxuan/master_thesis/KGRules/notebooks


## Extract paths from KG via Cypher Queries
+ direct_increase_proteins: query = `MATCH p = (n:Protein)-[r:increases]->(m:Pathology) where m.id contains 'Alzheimer Disease' return p`
+ direct_increase_genes: `MATCH p = (n:Gene)-[r:increases]->(m:Pathology) where m.id contains 'Alzheimer Disease' return p`
+ direct_decrease_proteins: `MATCH p = (n:Protein)-[r:decreases]->(m:Pathology) where m.id contains 'Alzheimer Disease' return p`
+ direct_decrease_genes: `MATCH p = (n:Gene)-[r:decreases]->(m:Pathology) where m.id contains 'Alzheimer Disease' return p`
+ len2_path_toAD:`match p = (src)-[r1]-(mid)-[r2]-(dst) where dst.id contains 'Alzheimer Disease' and (src:Protein or mid:Protein) return p`

In [ ]:
query_increase = """
    MATCH (s)-[r:increases]->(o)
    WHERE (s:Protein OR s:Gene) AND o.id contains 'Alzheimer Disease'
    RETURN s.id AS subject, type(r) AS predicate, o.id AS object
    """
query_decrease = """
    MATCH (s)-[r:decreases]->(o)
    WHERE (s:Protein OR s:Gene) 
    RETURN s.id AS subject, type(r) AS predicate, o.id AS object
    """
query_path2 = """
    MATCH (s)-[r1]-(m)-[r2]->(o)
    WHERE (s:Protein Or m:Protein Or o:Protein) AND (r2:increases OR r2:decreases)
    RETURN s.id AS subject, type(r1) AS predicate1, m.id as middle,type(r2) as predicate2, o.id AS object
    """
query_path3 = """
    MATCH (s)-[r1]-(n)-[r2]-(m)-[r3]->(o)
    WHERE (s:Protein OR m:Protein OR n:Protein) AND (o.id contains 'Alzheimer Disease') AND (r3:increases OR r3:decreases)
    RETURN s.id AS subject, type(r1) AS predicate1, n.id as middle1,type(r2) as predicate2, m.id as middle2, type(r3) as predicate3, o.id AS object
    """


In [ ]:
def get_paths_by_cypher(query):
    def get_triples(tx, query=query):
        # This query handles the 'length 2' and 'contains protein' logic we discussed
        result = tx.run(query)
        return [record.data() for record in result]
    
    driver = GraphDatabase.driver("bolt://localhost:7690", auth=('neo4j','test1237'))
    with driver.session() as session:
        triples_data = session.execute_read(get_triples)

    # 3. Convert to DataFrame for easy viewing in VS Code
    df_triples = pd.DataFrame(triples_data)

    # 4. Save to CSV so you can open it in VS Code's CSV viewer
    #df_triples.to_csv("extracted_triples.csv", index=False)

    print(f"Extracted {len(df_triples)} triples.")
    return df_triples

In [102]:
df_increase = get_paths_by_cypher(query_increase)
df_decrease = get_paths_by_cypher(query_decrease)
df_path2 = get_paths_by_cypher(query_path2)
df_path3 = get_paths_by_cypher(query_path3)

Extracted 27 triples.
Extracted 4 triples.
Extracted 995 triples.
Extracted 23820 triples.


In [103]:
df_path2

,subject,predicate1,middle,predicate2,object
0,"p(HGNC:""APP"",frag(""672_713""))",decreases,"composite(p(HGNC:""NOS3""),p(HGNC:""PIN1""))",decreases,"path(MESH:""Alzheimer Disease"")"
1,"deg(p(HGNC:""APP"",frag(""672_713"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
2,"p(HGNC:""PLG"")",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
3,"deg(p(HGNC:""APP"",frag(""672_711"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
4,"p(HGNC:""APP"",frag(""672_713""))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
...,...,...,...,...,...
990,"p(HGNC:""ALB"")",regulates,"a(CHEBI:""amyloid-beta"")",increases,"path(MESH:""Alzheimer Disease"")"
991,"p(HGNC:""APP"",frag(""672_713""))",is_a,"a(CHEBI:""amyloid-beta"")",increases,"path(MESH:""Alzheimer Disease"")"
992,"p(HGNC:""APP"",frag(""672_711""))",is_a,"a(CHEBI:""amyloid-beta"")",increases,"path(MESH:""Alzheimer Disease"")"
993,"p(HGNC:""MTHFR"",var(""Q,677,A""))",increases,"g(HGNC:""MTHFR"",var(""C,677,T""))",increases,"path(MESH:""Alzheimer Disease"")"


In [98]:
df_path3['predicate1'].unique()

array(['decreases', 'increases', 'regulates', 'is_a',
       'directly_decreases', 'directly_increases'], dtype=object)

In [66]:
len(df_path3['middle2'].unique())
df_path3[df_path3['middle2'].str.startswith('p(')]

,subject,predicate1,middle1,predicate2,middle2,predicate3,object
428,"a(MESH:""Microglia"")",increases,"deg(p(HGNC:""APP"",frag(""672_713"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
429,"complex(p(HGNC:""APP"",frag(""672_713"")),p(HGNC:""...",increases,"deg(p(HGNC:""APP"",frag(""672_713"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
430,"a(MESH:""Lysosomes"")",increases,"deg(p(HGNC:""APP"",frag(""672_713"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
431,"a(MESH:""Astrocytes"")",increases,"deg(p(HGNC:""APP"",frag(""672_713"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
432,"act(p(HGNC:""IL4""))",increases,"deg(p(HGNC:""APP"",frag(""672_713"")))",increases,"p(HGNC:""PLAT"")",decreases,"path(MESH:""Alzheimer Disease"")"
...,...,...,...,...,...,...,...
17697,"p(HGNC:""CHI3L1"")",decreases,"act(p(HGNC:""NFKB1""))",increases,"p(HGNC:""AGER"")",increases,"path(MESH:""Alzheimer Disease"")"
17698,"p(HGNC:""TLR4"")",increases,"act(p(HGNC:""NFKB1""))",increases,"p(HGNC:""AGER"")",increases,"path(MESH:""Alzheimer Disease"")"
17699,"bp(GO:""neuron death"")",decreases,"act(p(HGNC:""NFKB1""))",increases,"p(HGNC:""AGER"")",increases,"path(MESH:""Alzheimer Disease"")"
17700,"p(HGNC:""CHI3L1"")",decreases,"act(p(HGNC:""NFKB2""))",increases,"p(HGNC:""AGER"")",increases,"path(MESH:""Alzheimer Disease"")"


In [49]:
expression_df = pd.read_csv('../Anyburl/data/adni_gene_cleaned.csv', index_col=0).T
expression_df

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,3.651,2.2865,3.039,2.3395,2.783,2.25300,2.081,7.043,3.30950,2.202,...,5.748,3.77150,4.0365,4.353000,5.1763,1.9045,5.31750,9.2190,7.3010,5.7490
037_S_4410,3.183,2.1230,3.543,2.2085,2.383,2.37550,1.733,6.773,3.27625,2.317,...,5.974,4.17300,4.4415,4.520667,5.0964,1.9265,5.38800,8.3785,6.7580,6.0935
006_S_4153,3.278,2.3545,3.528,2.1745,2.593,2.46825,1.841,6.910,3.20875,2.540,...,5.119,3.91275,4.6210,4.230667,5.1143,2.2315,5.62800,9.1085,7.3365,5.2615
116_S_1232,3.371,2.3725,3.835,2.1545,2.570,2.51925,2.249,7.209,3.24950,2.559,...,4.904,3.73800,4.4435,4.050667,5.1520,2.0545,5.46300,9.3210,7.1685,4.7340
099_S_4205,3.358,2.3865,3.392,2.1720,2.660,2.39725,1.893,6.920,3.15425,2.347,...,5.533,3.94600,4.5215,4.639667,5.2103,2.0405,5.64750,9.0300,7.2025,5.4575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
009_S_2381,3.302,2.5075,3.524,2.2525,2.876,2.76275,2.089,6.805,3.09150,2.650,...,4.523,3.50150,4.4685,3.962333,5.0892,1.9320,5.44275,9.2900,6.7035,4.7915
053_S_4557,3.403,2.3090,3.515,2.3225,3.106,2.85125,2.102,7.265,3.27575,2.603,...,5.087,3.58325,4.1555,4.125667,5.0986,1.9445,5.14000,9.8090,7.2810,4.7055
073_S_4300,3.530,2.4155,3.651,2.0760,2.707,2.38325,2.092,7.375,3.26950,2.557,...,4.938,3.63450,4.5165,4.070333,5.1731,2.0795,5.38775,9.5215,6.9825,5.0785
041_S_4014,3.532,2.4545,3.609,2.3495,3.081,2.69125,2.024,7.257,3.11425,2.497,...,4.037,3.68525,4.0480,3.976333,4.8241,2.0075,5.05725,9.5810,6.6865,3.8660


## Grouping paths 
### Based on Biological rule of entities

+ amyloid related paths
+ ...

### Based on Clustering of all path scores

*** Attention:
+ abundance, degredation nodes also contains proteins and genes names, should i take them into account when calculating path-scores using expression data?
+ The follwoing code didn't take them into account.

In [139]:
relation_signs = {
    "increases": +1.0,
    "decreases": -1.0,
    "directly_decreases": +1.0,
    "directly_increases":-1.0,
    "regulates": +0.5,
    "is_a":  +1.0,
}
def extract_path_from_row(row):
    path = []
    values = row.dropna().tolist()
    # iterate in steps of 2 to grab (node, relation) pairs
    # stop before the last element
    for i in range(0, len(values) - 1, 2):
        head = values[i]
        relation = values[i+1]
        path.append((head, relation))
    return path

def extract_hgnc_name(text, pattern:str=r'(\(HGNC:"(\w+)"\))'):
    if text.startswith('p') or text.startswith('g'):
        match = re.search(pattern, text)
        if match:
            name = match.group(2)
            #print(name)
            return name
        else:
            return None

def compute_all_path_scores(expr_df, path_dfs):
    
    df_all_scores = pd.DataFrame()
    for df_name, df in path_dfs.items():
        path_scores = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Iter path rows"):
            path = extract_path_from_row(row)  # list of (head, relation, tail)
            #print(path)
            score = pd.Series([1.0] * len(expr_df), index=expr_df.index)
            for (head, relation) in path:
                head_name = extract_hgnc_name(head)
                #print(head, head_name)
                if head_name in expr_df.columns:
                    score *= expr_df[head_name]
                    #print('score without sign:', score)
                    score *= relation_signs.get(relation)
                    #print(head, relation)
                    path_scores.append(score)
                    #print('score with sign:',score)
                else:
                    #print(f"Skipping {head} because not in Expression_df.")
                    continue
            
        df_scores = pd.concat(path_scores, axis=1)
        df_all_scores = pd.concat([df_all_scores, df_scores], axis=1) 
    return df_all_scores
    

In [140]:
from sklearn.cluster import KMeans

def cluster_path_aggregation(path_dfs, expression_df, output:str, n_clusters = 20):
    
    # 1. compute all  individual path scores
    df_all_path_scores = compute_all_path_scores(expression_df, path_dfs)
    path_score_matrix = df_all_path_scores.values
    # shape: [n_patients, n_paths]

    # 2. cluster paths into k groups
    kmeans = KMeans(n_clusters=n_clusters)
    path_clusters = kmeans.fit(path_score_matrix.T)  # cluster paths
    
    # 3. Aggregate within each cluster
    labels = kmeans.labels_
    features = []

    for i in range(20):
        # Find the indices of the items belonging to cluster i
        indices = [idx for idx, label in enumerate(labels) if label == i]
        
        # Slice the original matrix (selecting the columns in this cluster)
        cluster_matrix = path_score_matrix[:, indices]
        #print('Cluster matrix shape:', cluster_matrix.shape)
        feature = cluster_matrix.mean(axis=1)
        #print('feature shape:', feature.shape)
        features.append(feature)
    df_knn_features = pd.DataFrame(features).T
    df_knn_features.to_csv(output, sep='\t')
    return df_knn_features, path_score_matrix, kmeans

df_knn_features, scores_matrix, kmeans= cluster_path_aggregation(
    {'len2_paths':df_path2, 'len3_paths':df_path3},
    expression_df,
    'knn_features.csv',
    20
) 

Iter path rows: 100%|██████████| 23820/23820 [00:11<00:00, 1985.87it/s]


## Convert paths to features

In [142]:
def compute_features_for_patient(expr_df, path_dfs, relation_signs, output):
    df_features = pd.DataFrame()

    for df_name, df in path_dfs.items():
        path_scores = []

        for _, row in df.iterrows():
            path = extract_path_from_row(row)  # list of (head, relation, tail)
            #print(path)
            score = pd.Series([1.0] * len(expr_df), index=expr_df.index)
            for (head, relation) in path:
                head_name = extract_hgnc_name(head)
                #print(head, head_name)
                if head_name in expr_df.columns:
                    score *= expr_df[head_name]
                    #print('score without sign:', score)
                    score *= relation_signs.get(relation)
                    #print('relation:', relation)
                    #print('score with sign:',score)
                else:
                    print(f"Skipping {head} because not in Expression_df.")
            path_scores.append(score)
        df_scores = pd.concat(path_scores, axis=1)
        # Aggregate
        # df_features[f"{df_name}_positive"] = df_scores.clip(lower=0).sum(axis=1)
        # df_features[f"{df_name}_negative"] = df_scores.clip(upper=0).sum(axis=1)
        df_features[f"{df_name}_net"] = df_scores.sum(axis=1)

        #break
    df_features.to_csv(output, sep='\t')
    return df_features

In [144]:
path_dfs = {
    'direct_increase_AD': df_increase,
    'direct_decrease_AD': df_decrease
            }

df_direct_nodes = compute_features_for_patient(expression_df, path_dfs, relation_signs, 'direct_features.csv')
df_direct_nodes

Skipping g(HGNC:"MTHFR",var("C,677,T")) because not in Expression_df.
Skipping g(HGNC:"RFC1",var("G,80,A")) because not in Expression_df.
Skipping p(HGNC:"MAPT",pmod(Ph)) because not in Expression_df.
Skipping p(HGNC:"APP",frag("672_713")) because not in Expression_df.
Skipping p(HGNC:"APP",var("N,10,Y")) because not in Expression_df.
Skipping p(MGI:"Bace1") because not in Expression_df.
Skipping p(MGI:"Mapt",pmod(Ph)) because not in Expression_df.
Skipping p(HGNC:"APP",var("A,713,T")) because not in Expression_df.
Skipping p(HGNC:"APP",frag("1_8")) because not in Expression_df.
Skipping p(HGNC:"APP",frag("9_16")) because not in Expression_df.
Skipping p(HGNC:"APP",frag("1_16")) because not in Expression_df.
Skipping p(HGNC:"MTHFR",var("A,677,V")) because not in Expression_df.
Skipping p(HGNC:"H3F3A",var("P,T,231")) because not in Expression_df.
Skipping p(HGNC:"H3F3A",pmod(Ub)) because not in Expression_df.
Skipping p(RGD:"Mapt",pmod(Ph)) because not in Expression_df.
Skipping g(HGNC:

,direct_increase_AD_net,direct_decrease_AD_net
116_S_1249,83.551529,-13.746381
037_S_4410,83.031632,-14.690714
006_S_4153,82.303691,-14.079095
116_S_1232,83.629590,-14.440381
099_S_4205,83.280137,-15.148429
...,...,...
009_S_2381,82.599023,-14.988810
053_S_4557,84.311978,-14.787524
073_S_4300,84.246519,-14.026286
041_S_4014,81.979936,-15.031952


### Extract paths via Networkx

In [46]:
def extract_paths_to_disease(G, disease_node='path(MESH:"Alzheimer Disease")', max_depth=2):
    """
    Extract all paths from any entity to AD node
    """
    paths = []
    for node in G.nodes():
        if node == disease_node:
            continue
        # Find all simple paths to disease within max depth
        for path in nx.all_simple_paths(G, node, disease_node, cutoff=max_depth):
            #print(path)
            # Store path with its edge relations
            edge_path = []
            for i in range(len(path)-1):
                if path[i].startswith('g') or path[i].startswith('p'):
                    relation = list(G[path[i]][path[i+1]].keys())[0]
                    #print(relation)
                    #break
                    edge_path.append((path[i], relation, path[i+1]))
            if len(edge_path)>0:
                paths.append(edge_path)
    print(f"Found {len(paths)} paths to Node {disease_node}")
    return paths

In [47]:
ad_kg_path = "../Anyburl/data/KG/ad_noreverse_causal.pkl"
ad_kg = load_graph(ad_kg_path)
paths = extract_paths_to_disease(ad_kg)

Graph has 3732 nodes and 5277 edges.
Found 639 paths to Node path(MESH:"Alzheimer Disease")


In [48]:
paths

[[('g(HGNC:"BDNF")', 'decreases', 'p(HGNC:"APP",frag("672_713"))'),
  ('p(HGNC:"APP",frag("672_713"))',
   'increases',
   'path(MESH:"Alzheimer Disease")')],
 [('g(HGNC:"PPARG")', 'decreases', 'p(HGNC:"APP",frag("672_713"))'),
  ('p(HGNC:"APP",frag("672_713"))',
   'increases',
   'path(MESH:"Alzheimer Disease")')],
 [('g(HGNC:"PPARG")', 'decreases', 'bp(GO:"inflammatory response")')],
 [('g(HGNC:"PPARG")', 'increases', 'act(g(HGNC:"APOE"),ma(tscript))')],
 [('g(HGNC:"APP")', 'increases', 'p(HGNC:"APP")'),
  ('p(HGNC:"APP")', 'increases', 'path(MESH:"Alzheimer Disease")')],
 [('g(HGNC:"APP")', 'increases', 'a(CHEBI:"amyloid-beta")')],
 [('g(HGNC:"NR1H3")', 'increases', 'act(g(HGNC:"APOE"),ma(tscript))')],
 [('g(HGNC:"APOE")', 'increases', 'a(CHEBI:"amyloid-beta")')],
 [('g(HGNC:"CR1")', 'directly_increases', 'path(MESH:"Alzheimer Disease")')],
 [('g(HGNC:"APP",var("G,275341,C"))',
   'increases',
   'p(HGNC:"APP",frag("672_713"))'),
  ('p(HGNC:"APP",frag("672_713"))',
   'increases',
